# 04 — Causal discovery experiments

## Default configurations
- Asia: CausalNex=NOTEARS, Causal-Learn=PC, CDT=PC
- Elderly: CausalNex=NOTEARS, Causal-Learn=PC, CDT=PC

## Best configurations
- Asia: CausalNex=NOTEARS, Causal-Learn=GES, CDT=PC
- Elderly: CausalNex=HillClimbing, Causal-Learn=PC, CDT=GES

The best configuration is selected retrospectively using the reference graph and
therefore represents a controlled benchmarking condition, not practical algorithm
selection when the true causal graph is unknown.


In [ ]:
# CAUSAL DISCOVERY EXPERIMENTS

import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import pyAgrum as gum



# CONFIGURATION

OUTPUT_DIR = "04_Causal_Discovery_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# Original observational datasets

AsiaData = pd.read_csv("AsiaData.csv")
ElderlyData = pd.read_csv("ElderlyData.csv")



# Reference CBNs
# These networks are NOT used to learn the structure.
# They are only used afterwards for evaluation.

AsiaBN = gum.loadBN("AsiaBN.bif")
ElderlyBN = gum.loadBN("ElderlyBN.bif")



def prepare_data(data):
    """
    Prepare the observational dataset for causal discovery.
    """

    data = data.copy()

    # Keep only complete observations
    data = data.dropna()

    # Convert categorical variables to numerical values
    for column in data.columns:

        if not pd.api.types.is_numeric_dtype(data[column]):

            data[column] = pd.factorize(
                data[column]
            )[0]

    return data


AsiaData = prepare_data(AsiaData)
ElderlyData = prepare_data(ElderlyData)


# 1. CAUSALNEX — NOTEARS

def causalnex_notears(data):

    from causalnex.structure.notears import from_pandas

    structure_model = from_pandas(data)

    return list(structure_model.edges())


# 2. CAUSALNEX — HILL CLIMBING

def causalnex_hillclimbing(data):

    from causalnex.structure import StructureModel
    from causalnex.structure.notears import from_pandas

    # --------------------------------------------------------
    # NOTE:
    # CausalNex's main structure-learning API is NOTEARS.
    # Hill Climbing is handled below through pgmpy.
    # --------------------------------------------------------

    from pgmpy.estimators import HillClimbSearch

    hc = HillClimbSearch(data)

    model = hc.estimate()

    return list(model.edges())


# 3. CAUSAL-LEARN — PC

def causallearn_pc(data):

    from causallearn.search.ConstraintBased.PC import pc

    X = data.values.astype(float)

    cg = pc(
        X,
        alpha=0.05
    )

    graph = cg.G

    columns = list(data.columns)

    edges = []

    for i in range(len(columns)):

        for j in range(len(columns)):

            if i == j:
                continue

            node_i = graph.nodes[i]
            node_j = graph.nodes[j]

            if graph.is_adjacent_to(
                node_i,
                node_j
            ):

                endpoint_ij = graph.get_endpoint(
                    node_i,
                    node_j
                )

                endpoint_ji = graph.get_endpoint(
                    node_j,
                    node_i
                )

                # Directed edge i -> j
                if (
                    endpoint_ij.name == "TAIL"
                    and
                    endpoint_ji.name == "ARROW"
                ):

                    edges.append(
                        (
                            columns[i],
                            columns[j]
                        )
                    )

    return edges


# 4. CAUSAL-LEARN — GES

def causallearn_ges(data):

    from causallearn.search.ScoreBased.GES import ges

    X = data.values.astype(float)

    result = ges(X)

    graph = result["G"]

    columns = list(data.columns)

    edges = []

    for i in range(len(columns)):

        for j in range(len(columns)):

            if i == j:
                continue

            node_i = graph.nodes[i]
            node_j = graph.nodes[j]

            if graph.is_adjacent_to(
                node_i,
                node_j
            ):

                endpoint_ij = graph.get_endpoint(
                    node_i,
                    node_j
                )

                endpoint_ji = graph.get_endpoint(
                    node_j,
                    node_i
                )

                if (
                    endpoint_ij.name == "TAIL"
                    and
                    endpoint_ji.name == "ARROW"
                ):

                    edges.append(
                        (
                            columns[i],
                            columns[j]
                        )
                    )

    return edges


# 5. CDT — PC

def cdt_pc(data):

    import cdt

    model = cdt.causality.graph.PC()

    graph = model.create_graph_from_data(
        data
    )

    return list(graph.edges())


# 6. CDT — GES

def cdt_ges(data):

    import cdt

    model = cdt.causality.graph.GES()

    graph = model.create_graph_from_data(
        data
    )

    return list(graph.edges())


# VISUALIZATION

def save_structure_image(
    edges,
    nodes,
    filename,
    title
):

    graph = nx.DiGraph()

    graph.add_nodes_from(nodes)

    graph.add_edges_from(edges)

    plt.figure(
        figsize=(12, 9)
    )

    pos = nx.spring_layout(
        graph,
        seed=42
    )

    nx.draw(
        graph,
        pos,
        with_labels=True,
        node_size=2500,
        arrows=True,
        font_size=9
    )

    plt.title(title)

    plt.tight_layout()

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


# EXPERIMENT CONFIGURATIONS

DEFAULT_CONFIG = {

    "Asia": {

        "CausalNex": (
            "NOTEARS",
            causalnex_notears
        ),

        "Causal-Learn": (
            "PC",
            causallearn_pc
        ),

        "CDT": (
            "PC",
            cdt_pc
        )
    },

    "Elderly": {

        "CausalNex": (
            "NOTEARS",
            causalnex_notears
        ),

        "Causal-Learn": (
            "PC",
            causallearn_pc
        ),

        "CDT": (
            "PC",
            cdt_pc
        )
    }
}


BEST_CONFIG = {

    "Asia": {

        "CausalNex": (
            "NOTEARS",
            causalnex_notears
        ),

        "Causal-Learn": (
            "GES",
            causallearn_ges
        ),

        "CDT": (
            "PC",
            cdt_pc
        )
    },

    "Elderly": {

        "CausalNex": (
            "HillClimbing",
            causalnex_hillclimbing
        ),

        "Causal-Learn": (
            "PC",
            causallearn_pc
        ),

        "CDT": (
            "GES",
            cdt_ges
        )
    }
}



def run_experiment(
    dataset_name,
    data,
    configurations,
    experiment_name
):

    print("\n")
    print("=" * 70)
    print(
        dataset_name,
        "|",
        experiment_name
    )
    print("=" * 70)

    nodes = list(data.columns)

    results = {}

    for framework, (
        algorithm_name,
        algorithm
    ) in configurations[dataset_name].items():

        print(
            "\nRunning:",
            framework,
            "-",
            algorithm_name
        )

        try:

            edges = algorithm(data)

            results[framework] = {

                "algorithm": algorithm_name,

                "edges": edges
            }

            filename = os.path.join(

                OUTPUT_DIR,

                f"{dataset_name}_"
                f"{experiment_name}_"
                f"{framework}_"
                f"{algorithm_name}.png"
            )

            save_structure_image(

                edges,

                nodes,

                filename,

                (
                    f"{dataset_name} - "
                    f"{framework} - "
                    f"{algorithm_name}"
                )
            )

            print(
                "Number of directed edges:",
                len(edges)
            )

            print(
                "Saved:",
                filename
            )

        except Exception as e:

            print(
                f"ERROR - {framework} "
                f"{algorithm_name}:",
                e
            )

    return results


# 04.1 — DEFAULT CONFIGURATIONS

Asia_Default = run_experiment(

    "Asia",

    AsiaData,

    DEFAULT_CONFIG,

    "Default"
)


Elderly_Default = run_experiment(

    "Elderly",

    ElderlyData,

    DEFAULT_CONFIG,

    "Default"
)


# 04.2 — BEST CONFIGURATIONS

Asia_Best = run_experiment(

    "Asia",

    AsiaData,

    BEST_CONFIG,

    "Best"
)


Elderly_Best = run_experiment(

    "Elderly",

    ElderlyData,

    BEST_CONFIG,

    "Best"
)


For each run, save the learned edge list, execution time and evaluation metrics.
Do not insert fabricated numerical results.
